# 模块二：用户价值与行为探究 (User Value & Behavior Analysis)

**业务背景调整声明：**
鉴于 Olist 开源数据集仅包含交易后（Post-transaction）履约数据，缺乏浏览 (`view`) 和加购 (`cart`) 等前端行为埋点，构建传统前端转化漏斗存在“无米之炊”的问题。
基于数据驱动务实的原则，本模块将分析重心由“前端转化”调整为**“存量用户价值挖掘”**。

**本模块核心目标：**
1. 构建高度聚合的分析宽表 (Analytical Base Table, ABT)，避免表连接过程中的金额膨胀。
2. 产出**月度同期群留存矩阵 (Cohort Analysis)**，洞察用户生命周期规律。
3. 挖掘**跨品类购物篮关联 (Market Basket Analysis)**，为交叉销售 (Cross-selling) 提供策略支撑。

In [1]:
# 1. 开启自动重载魔术命令
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 将上一级目录加入系统路径
sys.path.append('../')
from src.user_value import UserValueAnalyzer

# 2. 加载模块一产出的干净数据 (Clean Data) 和剩余的维度表 (Dimension Tables)
print("Loading processed and raw datasets...")
df_orders_clean = pd.read_parquet("../data/processed/olist_orders_clean.parquet")
df_payments_clean = pd.read_parquet("../data/processed/olist_payments_clean.parquet")
df_items_clean = pd.read_parquet("../data/processed/olist_items_clean.parquet")

df_customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
df_products = pd.read_csv("../data/raw/olist_products_dataset.csv")

print("✅ Data successfully loaded.")

Loading processed and raw datasets...
✅ Data successfully loaded.


## 步骤一：构建底层分析宽表 (ABT)
此步骤的核心难点在于：订单表、支付表（一单多笔支付）、商品表（一单多件商品）之间存在 1对多 关系。
如果在连接时不先进行聚合，会导致订单总金额 (Monetary) 发生笛卡尔积膨胀。

In [2]:
# 3. 实例化用户价值引擎
analyzer = UserValueAnalyzer(
    df_orders=df_orders_clean,
    df_payments=df_payments_clean,
    df_customers=df_customers,
    df_items=df_items_clean,
    df_products=df_products
)

# 4. 构建防膨胀宽表
print("Building Analytical Base Table (ABT)...")
df_master = analyzer.build_analytical_base_table()

print(f"ABT Shape: {df_master.shape}")
display(df_master.head(3))

Building Analytical Base Table (ABT)...
ABT Shape: (110710, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_dt,purchase_month,...,customer_state,payment_value,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_weight_g
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10-02 10:56:33,2017-10,...,SP,38.71,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,500.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07-24 20:41:37,2018-07,...,BA,141.46,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,400.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08-08 08:38:49,2018-08,...,GO,179.12,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,420.0


## 步骤二：同期群留存分析 (Cohort Analysis)
计算按首单月份划分的用户群，在后续月份的复购留存率。
*业务洞察预警：电商大促拉新的用户，长期留存表现通常如何？*

In [3]:
# 5. 计算并展示同期群矩阵
cohort_matrix = analyzer.calculate_cohort_matrix()

# 面试展示亮点：直接在 Pandas 内利用 background_gradient 渲染热力图
# 这样不仅代码简洁，视觉冲击力也极强
print("✅ 同期群留存热力矩阵：")
display(cohort_matrix.style.background_gradient(cmap='YlGnBu', axis=None, vmin=0, vmax=1))

# 从热力图中可以看出，Olist 平台的复购率极低（普遍不足 1%），属于典型的“一锤子买卖”平台。
# 业务建议：重点应放在提升客单价（连带率）上，而非盲目投入预算做复购唤醒。

✅ 同期群留存热力矩阵：


cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
first_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-10,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.340000,0.000000,0.000000,0.340000,0.000000,0.340000,0.000000,0.340000,0.000000,0.340000,0.000000,0.340000,0.680000,0.340000
2016-12,100.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2017-01,100.000000,0.400000,0.270000,0.130000,0.400000,0.130000,0.400000,0.130000,0.130000,0.000000,0.270000,0.130000,0.670000,0.400000,0.130000,0.130000,0.270000,0.400000,0.130000,0.000000
2017-02,100.000000,0.240000,0.300000,0.120000,0.410000,0.120000,0.240000,0.180000,0.120000,0.180000,0.120000,0.300000,0.120000,0.180000,0.120000,0.060000,0.060000,0.240000,0.000000,0.000000
2017-03,100.000000,0.500000,0.350000,0.390000,0.350000,0.160000,0.160000,0.310000,0.350000,0.080000,0.390000,0.120000,0.230000,0.120000,0.160000,0.230000,0.080000,0.160000,0.000000,0.000000
2017-04,100.000000,0.560000,0.220000,0.170000,0.350000,0.260000,0.350000,0.300000,0.300000,0.170000,0.220000,0.040000,0.090000,0.040000,0.090000,0.090000,0.170000,0.000000,0.000000,0.000000
2017-05,100.000000,0.450000,0.480000,0.370000,0.310000,0.340000,0.430000,0.110000,0.200000,0.230000,0.260000,0.310000,0.260000,0.030000,0.200000,0.230000,0.000000,0.000000,0.000000,0.000000
2017-06,100.000000,0.450000,0.360000,0.390000,0.260000,0.360000,0.390000,0.230000,0.130000,0.230000,0.320000,0.320000,0.160000,0.130000,0.190000,0.000000,0.000000,0.000000,0.000000,0.000000


## 步骤三：购物篮分析与模型输出
既然复购率低，提升单次购买的连带率就成了营收关键。我们通过购物篮分析寻找经常被一起购买的商品品类。

In [4]:
# 6. 计算跨品类交叉购买规则
basket_rules = analyzer.calculate_market_basket()

# 整理为更易读的 DataFrame 格式
df_basket_top10 = basket_rules.reset_index(name='co_occurrence_count').head(10)
df_basket_top10.columns = ['Product_A', 'Product_B', 'Co_Occurrence_Count']

print("✅ Top 10 跨品类搭售组合：")
display(df_basket_top10)

# 7. 缓存产出物，供模块三和模块四使用
df_master.to_parquet("../data/processed/olist_master_abt.parquet", index=False)
cohort_matrix.to_parquet("../data/processed/cohort_matrix.parquet")
print("Data saved successfully for Module 03 & 04.")

✅ Top 10 跨品类搭售组合：


,Product_A,Product_B,Co_Occurrence_Count
0,cama_mesa_banho,moveis_decoracao,68
1,moveis_decoracao,cama_mesa_banho,68
2,casa_conforto,cama_mesa_banho,42
3,cama_mesa_banho,casa_conforto,42
4,utilidades_domesticas,moveis_decoracao,23
5,moveis_decoracao,utilidades_domesticas,23
6,cama_mesa_banho,utilidades_domesticas,20
7,cool_stuff,bebes,20
8,utilidades_domesticas,cama_mesa_banho,20
9,bebes,cool_stuff,20


Data saved successfully for Module 03 & 04.
